In [ ]:
import pandas as pd
import numpy as np

from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests
from textblob import TextBlob

df = pd.read_csv("merged_event_logs_dialogue_only.csv")
df.head()

def sentiment(text):
    return TextBlob(str(text)).sentiment.polarity

df["sentiment"] = df["content"].apply(sentiment)

In [ ]:
from scipy.stats import mannwhitneyu, ttest_ind, shapiro

results = []

for dpy in sorted(df["days_per_year"].unique()):
    sub = df[df["days_per_year"] == dpy]

    base = sub[sub["condition"] == "baseline"]["sentiment"]
    trauma = sub[sub["condition"] == "trauma"]["sentiment"]

    if len(base) > 10 and len(trauma) > 10:
        # Normality test (Shapiro–Wilk)
        shapiro_base_p = shapiro(base).pvalue
        shapiro_trauma_p = shapiro(trauma).pvalue

        normal_base = shapiro_base_p > 0.05
        normal_trauma = shapiro_trauma_p > 0.05

        if normal_base and normal_trauma:
            # Parametric test
            stat, p = ttest_ind(base, trauma, equal_var=False)
            test_name = "Welch t-test"
        else:
            # Non-parametric test
            stat, p = mannwhitneyu(base, trauma, alternative="two-sided")
            test_name = "Mann–Whitney U"

        results.append({
            "comparison": f"baseline vs trauma ({dpy} day/year)",
            "test": test_name,
            "p_value": p,
            "median_baseline": base.median(),
            "median_trauma": trauma.median(),
            "shapiro_p_baseline": shapiro_base_p,
            "shapiro_p_trauma": shapiro_trauma_p,
            "baseline_normal": normal_base,
            "trauma_normal": normal_trauma
        })

res_df = pd.DataFrame(results)
res_df

In [ ]:
groups = []
labels = []

for dpy in sorted(df["days_per_year"].unique()):
    vals = df[df["days_per_year"] == dpy]["sentiment"]
    if len(vals) > 20:
        groups.append(vals)
        labels.append(f"{dpy} day/year")

h, p_kw = kruskal(*groups)

print("Kruskal–Wallis H =", round(h,3))
print("p-value =", p_kw)

In [ ]:
pvals = res_df["p_value"].values
reject, qvals, _, _ = multipletests(pvals, method="fdr_bh")

res_df["q_value"] = qvals
res_df["significant_FDR"] = reject

res_df

In [ ]:
summary = res_df[[
    "comparison",
    "median_baseline",
    "median_trauma",
    "p_value",
    "q_value",
    "significant_FDR"
]]

summary

In [ ]:
def cliffs_delta(x, y):
    """
    Computes Cliff's delta effect size.
    Returns delta in [-1, 1].
    """
    x = np.asarray(x)
    y = np.asarray(y)

    nx = len(x)
    ny = len(y)

    greater = 0
    less = 0

    for xi in x:
        greater += np.sum(xi > y)
        less += np.sum(xi < y)

    delta = (greater - less) / (nx * ny)
    return delta


In [ ]:
effect_results = []

for dpy in sorted(df["days_per_year"].unique()):
    sub = df[df["days_per_year"] == dpy]

    base = sub[sub["condition"] == "baseline"]["sentiment"]
    trauma = sub[sub["condition"] == "trauma"]["sentiment"]

    if len(base) > 10 and len(trauma) > 10:
        delta = cliffs_delta(base, trauma)

        effect_results.append({
            "comparison": f"baseline vs trauma ({dpy} day/year)",
            "cliffs_delta": delta,
            "abs_delta": abs(delta)
        })

effect_df = pd.DataFrame(effect_results)
effect_df

In [ ]:
def delta_category(d):
    ad = abs(d)
    if ad < 0.147:
        return "negligible"
    elif ad < 0.33:
        return "small"
    elif ad < 0.474:
        return "medium"
    else:
        return "large"

effect_df["magnitude"] = effect_df["cliffs_delta"].apply(delta_category)
effect_df

In [ ]:
final_table = pd.merge(
    summary,
    effect_df,
    on="comparison",
    how="left"
)

final_table